In [1]:
import pickle
import faiss
import os
from dotenv import load_dotenv
from google import genai

# Load corpus
with open("../data/all_chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

index = faiss.read_index("../data/all_index.faiss")

# Load the model (needed to embed queries)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

# Gemini client
load_dotenv("../.env")
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Search function
import numpy as np
def search(query, k=5):
    q_emb = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(q_emb, dtype="float32"), k)
    return [
        {"score": float(s), "text": all_chunks[i]["text"], "source": all_chunks[i]["source"]}
        for s, i in zip(scores[0], indices[0])
    ]

print("Loaded:", len(all_chunks), "chunks")
print("Index size:", index.ntotal)
print("Model ready:", model is not None)
print("Client ready:", client is not None)

/home/rgukt/Desktop/research-assistant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████| 103/103 [00:00<00:00, 482.21it/s]


Loaded: 1356 chunks
Index size: 1356
Model ready: True
Client ready: True


In [3]:
import time

def ask_with_retry(query, k=5, max_retries=5):
    """Call ask() with retry on 503 errors."""
    for attempt in range(max_retries):
        try:
            return ask(query, k=k)
        except Exception as e:
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                wait = 5 * (attempt + 1)  # 5s, 10s, 15s, 20s, 25s
                print(f"   503, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Failed after {max_retries} retries")

eval_results = []
for i, q in enumerate(eval_questions, 1):
    print(f"[{i}/{len(eval_questions)}] {q}")
    try:
        answer, retrieved = ask_with_retry(q)
        top_score = retrieved[0]["score"] if retrieved else 0
        top_source = retrieved[0]["source"] if retrieved else "none"
        eval_results.append({
            "question": q,
            "answer": answer,
            "top_score": top_score,
            "top_source": top_source,
        })
        print(f"   Score: {top_score:.3f} | {top_source}")
        print(f"   {answer[:150]}...")
    except Exception as e:
        print(f"   FAILED: {e}")
    print()
    time.sleep(3)  # gentle pacing to avoid rate limits

print(f"\nEvaluated {len(eval_results)} of {len(eval_questions)} questions")

[1/10] What is autoregressive retrieval augmentation?
   503, retrying in 5s...
   503, retrying in 10s...
   Score: 0.606 | 2506.06962v3.pdf
   Based on the provided context, autoregressive retrieval augmentation (specifically AR-RAG) is a patch-level or patch-based retrieval method designed f...

[2/10] How does MusT-RAG handle music question answering?
   503, retrying in 5s...
   Score: 0.705 | 2507.23334v2.pdf
   MusT-RAG handles music question answering (MQA) by adapting general-purpose LLMs to the music domain using a Retrieval-Augmented Generation (RAG) fram...

[3/10] What datasets were used to evaluate the AR-RAG paper?
   503, retrying in 5s...
   503, retrying in 10s...
   503, retrying in 15s...
   Score: 0.551 | 2506.06962v3.pdf
   I don't have enough information....

[4/10] What is the main limitation of standard RAG systems?
   503, retrying in 5s...
   Score: 0.517 | 2510.25621v1.pdf
   I don't have enough information....

[5/10] How does agentic RAG improve Quran ques